# RQ3 — Challenges and Support (Q21–Q22)

**Analysis to Address RQ3**: Q21–Q22 — main challenges to guarantee data reliability and frequency of support from other teams.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import utils as U

U.setup_matplotlib()
TABLES = U.DATA_PROC / "tables"
TABLES.mkdir(exist_ok=True)

df = U.load_anonymized()
print(f"N={len(df)}")


## Canonical options and helper functions

In [ ]:
Q21_OPTIONS = {
    "q21_inconsistency": [
        "Inconsistência entre diferentes fontes de dados",
        "Inconsistency between different data sources",
    ],
    "q21_incompleteness": ["Dados incompletos ou ausentes", "Incomplete or missing data"],
    "q21_no_standard": [
        "Falta de padronização nos formatos de dados",
        "Lack of standardization in data formats",
    ],
    "q21_outdated": ["Dados desatualizados ou não confiáveis", "Outdated or unreliable data"],
    "q21_collection_errors": [
        "Erros introduzidos durante a coleta e processamento",
        "Errors introduced during collection and processing",
    ],
    "q21_traceability": [
        "Dificuldade na rastreabilidade e versionamento dos dados",
        "Difficulty in data traceability and versioning",
    ],
    "q21_no_tools": [
        "Falta de ferramentas adequadas para validação da qualidade dos dados",
        "Lack of adequate tools for validating data quality",
    ],
}
Q21_LABELS = {
    "q21_inconsistency": "Inconsistency across sources",
    "q21_incompleteness": "Incomplete/missing data",
    "q21_no_standard": "Lack of format standardization",
    "q21_outdated": "Outdated/unreliable data",
    "q21_collection_errors": "Errors in collection and processing",
    "q21_traceability": "Difficulty in traceability/versioning",
    "q21_no_tools": "Lack of validation tools",
}


In [ ]:
rng = np.random.default_rng(42)

def _boot_ci(values, threshold=4, n_bootstrap=2000, ci_level=0.95):
    values = np.asarray(values)
    n = len(values)
    if n == 0:
        return float("nan"), float("nan")
    alpha = 1 - ci_level
    boot_indices = rng.integers(0, n, size=(n_bootstrap, n))
    boot_props = np.mean(values[boot_indices] >= threshold, axis=1)
    return np.percentile(boot_props, 100 * alpha / 2), np.percentile(boot_props, 100 * (1 - alpha / 2))

def parse_checkboxes(series, options):
    binary = pd.DataFrame(index=series.index, columns=list(options.keys()), dtype=bool)
    binary[:] = False
    residual = series.copy()
    for key, raw in options.items():
        labels = [raw] if isinstance(raw, str) else list(raw)
        present = pd.Series(False, index=series.index)
        for lab in labels:
            present = present | series.fillna("").str.contains(lab, regex=False)
            residual = residual.fillna("").str.replace(lab, "", regex=False)
        binary[key] = present
    residual = residual.str.replace(r"^[,\.\s]+|[,\.\s]+$", "", regex=True)
    residual = residual.where(residual.str.len() > 2, "")
    return binary, residual

def proportions_with_ci(binary, labels, n_total):
    rows = []
    for key in binary.columns:
        vals = binary[key].astype(int).values
        lo, hi = _boot_ci(vals, threshold=1)
        rows.append({
            "key": key, "label": labels[key],
            "n": int(binary[key].sum()),
            "pct": binary[key].mean() * 100,
            "ci_lo": lo * 100, "ci_hi": hi * 100,
        })
    return pd.DataFrame(rows).sort_values("pct", ascending=False).reset_index(drop=True)

def horizontal_bar(p, title, color, ax):
    p_sorted = p.sort_values("pct")
    y = np.arange(len(p_sorted))
    ax.barh(y, p_sorted["pct"], color=color, edgecolor="white")
    err_lo = np.clip(p_sorted["pct"] - p_sorted["ci_lo"], 0, None)
    err_hi = np.clip(p_sorted["ci_hi"] - p_sorted["pct"], 0, None)
    ax.errorbar(p_sorted["pct"], y, xerr=[err_lo, err_hi],
                fmt="none", ecolor="black", elinewidth=0.6, capsize=2)
    ax.set_yticks(y)
    ax.set_yticklabels(p_sorted["label"])
    ax.set_xlabel("% of responses")
    ax.set_xlim(0, 100)
    ax.set_title(title)
    for i, (pct, n) in enumerate(zip(p_sorted["pct"], p_sorted["n"])):
        ax.text(pct + 1, i, f"{int(n)}", va="center", fontsize=7)


## Q21 — Parsing and Proportions

In [ ]:
q21_bin, q21_res = parse_checkboxes(df["challenges_open"], Q21_OPTIONS)
n_q21 = df["challenges_open"].notna().sum()
print(f"Q21: n={n_q21}, average options marked = {q21_bin.sum(axis=1).mean():.2f}")

nonempty = q21_res[q21_res.str.len() > 0]
if len(nonempty):
    print("Q21 residuals:")
    for idx, txt in nonempty.items():
        print(f"  P{idx:02d}: {txt!r}")


In [ ]:
p21 = proportions_with_ci(q21_bin, Q21_LABELS, n_q21)
p21


## Q21 — Main Challenges

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7.0, 4.0))
horizontal_bar(p21, f"Q21 — Main challenges to ensure reliability (n={n_q21})", U.PALETTE_WONG[6], ax)
fig.tight_layout()
U.save_fig(fig, "challenges_q21")
plt.show()


## Q22 — Support Frequency from Other Teams

In [ ]:
supp_labels = ["Rarely", "Occas.", "Often", "Always"]

fig, ax = plt.subplots(figsize=(5.0, 2.8))
col, labels, title = (
    "support_freq", supp_labels,
    "Q22 — Support frequency from other teams"
)
counts = df[col].value_counts(dropna=False).reindex(range(1, len(labels) + 1), fill_value=0)
pct = counts / counts.sum() * 100
ax.bar(range(len(labels)), pct.values, color=U.PALETTE_WONG[2], edgecolor="white")
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=30, ha="right")
ax.set_ylabel("% of responses")
ax.set_title(title)
ax.set_ylim(0, max(pct.values) * 1.15)
for i, v in enumerate(pct.values):
    ax.text(i, v + 1, f"{v:.0f}%", ha="center", va="bottom", fontsize=7)
fig.tight_layout()
U.save_fig(fig, "frequency_q22")
plt.show()


## LaTeX Table and Key Findings

In [ ]:
def render_latex(p: pd.DataFrame, label: str, n: int) -> list[str]:
    header = "\\multicolumn{4}{l}{\\textit{" + label + f" (n={n})" + "}} \\\\"
    out = [header]
    for _, r in p.iterrows():
        out.append(f"\\quad {r['label']} & {int(r['n'])} & {r['pct']:.0f}\\% & [{r['ci_lo']:.0f}--{r['ci_hi']:.0f}] \\\\")
    return out

lines = [
    "\\begin{table}[t]",
    "\\caption{Challenges to ensure data reliability. Proportions with Bootstrap 95\\% CI.}",
    "\\label{tab:challenges-rq3}",
    "\\centering",
    "\\small",
    "\\begin{tabular}{lrrr}",
    "\\toprule",
    "\\textbf{Item} & \\textbf{n} & \\textbf{\\%} & \\textbf{IC 95\\%} \\\\",
    "\\midrule",
]
lines.extend(render_latex(p21, "Q21. Challenges to ensure reliability", n_q21))
lines.extend(["\\bottomrule", "\\end{tabular}", "\\end{table}"])
(TABLES / "challenges_rq3.tex").write_text("\n".join(lines))
print("[saved] tables/challenges_rq3.tex")


In [ ]:
print("--- Q21 challenges ---")
for _, r in p21.head(3).iterrows():
    print(f"  {r['label']}: {r['pct']:.0f}% (IC95 {r['ci_lo']:.0f}-{r['ci_hi']:.0f})")

q22_counts = df["support_freq"].value_counts(dropna=False).sort_index()
pct_occasional = (df["support_freq"] <= 2).sum() / df["support_freq"].notna().sum() * 100
print(f"--- Q22 support ---")
print(f"  Rarely/occasionally: {pct_occasional:.1f}%")
